In [2]:
# Colab cell

import pandas as pd

# Load
teams_df = pd.read_csv('/content/teams.csv')

# Show preview
print(teams_df['total_capacity'].head(10))

# Cleaning function
def fix_capacity(value):
    try:
        v_str = str(value)
        if '.' in v_str and value < 100:
            # treat it as a decimal misplaced, so shift 3 digits
            return float(v_str) * 1000
        else:
            return float(v_str)
    except:
        return None  # or np.nan

teams_df['total_capacity'] = teams_df['total_capacity'].apply(fix_capacity)

# check results
print(teams_df['total_capacity'].head(10))

# Save cleaned
teams_df.to_csv('/content/teams_capacity_cleaned.csv', index=False)


0    81.365
1    28.917
2    75.000
3    27.332
4    34.700
5    30.150
6    54.042
7    30.210
8    14.500
9    49.350
Name: total_capacity, dtype: float64
0    81365.0
1    28917.0
2    75000.0
3    27332.0
4    34700.0
5    30150.0
6    54042.0
7    30210.0
8    14500.0
9    49350.0
Name: total_capacity, dtype: float64


In [15]:

# move link column to the end
teams_df = teams_df[['team', 'stadium_name', 'total_capacity', 'lawn_heating', 'field_length', 'field_width', 'link']]

# remove duplicated rows based on 'link' column
teams_df = teams_df.drop_duplicates(subset=['link']).reset_index(drop=True)

# convert 'total_capacity' to numeric
# Use a smart parser that handles German thousands-separator dots (e.g. '32.100' -> 32100)
# without corrupting already-correct decimal values (e.g. '81365.0' -> 81365)
def parse_capacity(value):
    try:
        v_str = str(value).strip()
        # If value parses directly as a plain float, use it (handles '81365.0', '32100', etc.)
        plain = float(v_str)
        # If fractional part is zero it is already a whole number
        if plain == int(plain):
            return int(plain)
        # If it looks like a German thousands-separator (e.g. '32.100' has exactly 3 digits after the dot)
        import re
        if re.match(r'^\d{1,3}(\.\d{3})+$', v_str):
            return int(v_str.replace('.', ''))
        return int(plain)
    except (ValueError, TypeError):
        return None

teams_df['total_capacity'] = teams_df['total_capacity'].apply(parse_capacity)

# convert 'field_length' and 'field_width' to numeric, errors='coerce' will convert non-numeric values to NaN
teams_df['field_length'] = pd.to_numeric(
	teams_df['field_length'].astype(str).str.replace('m', '', regex=False).str.replace(' ', '', regex=False),
	errors='coerce'
)
teams_df['field_width'] = pd.to_numeric(
	teams_df['field_width'].astype(str).str.replace('m', '', regex=False).str.replace(' ', '', regex=False),
	errors='coerce'
)

# convert 'lawn_heating' to boolean
teams_df['lawn_heating'] = teams_df['lawn_heating'].str.lower().map({'ja': True, 'nein': False, '': None})

In [19]:
print(teams_df.head(10))
print(teams_df.shape)
print(teams_df.isna().sum())

                       team              stadium_name  total_capacity  \
0         Borussia Dortmund         SIGNAL IDUNA PARK        813650.0   
1             VfL Wolfsburg          Volkswagen Arena        289170.0   
2         FC Bayern München             Allianz Arena        750000.0   
3         Arminia Bielefeld               SchücoArena        273320.0   
4               SC Freiburg       Europa-Park Stadion        347000.0   
5       TSG 1899 Hoffenheim             PreZero Arena        301500.0   
6  Borussia Mönchengladbach  Stadion im Borussia-Park        540420.0   
7       Bayer 04 Leverkusen                  BayArena        302100.0   
8                 VfR Aalen              CENTUS Arena        145000.0   
9       1.FC Kaiserslautern      Fritz-Walter-Stadion        493500.0   

  lawn_heating  field_length  field_width  \
0         True         105.0         68.0   
1         True         105.0         68.0   
2         True         105.0         68.0   
3         True  

In [22]:
teams_df_nan= teams_df.isna()
teams_df_nan.index[teams_df_nan['total_capacity'] == True]

Index([35], dtype='int64')

In [24]:
print(teams_df.iloc[35,:])

team                                                SV Waldkirch
stadium_name                                       Elztalstadion
total_capacity                                               NaN
lawn_heating                                                 NaN
field_length                                                 NaN
field_width                                                  NaN
link              /sv-waldkirch/spielplan/verein/9487/saison_id/
Name: 35, dtype: object


In [44]:
teams_df.iloc[35,2]= 5000
teams_df.iloc[35,3]= False
print(teams_df.isna().sum())
print(teams_df.iloc[35,:])

teams_df['field_length'] = teams_df['field_length'].fillna(teams_df['field_length'].mean())
teams_df['field_width'] = teams_df['field_width'].fillna(teams_df['field_width'].mean())
teams_df.to_csv('/content/final_cleaned_teams.csv', index=False)

team              0
stadium_name      0
total_capacity    0
lawn_heating      0
field_length      0
field_width       0
link              0
dtype: int64
team                                                SV Waldkirch
stadium_name                                       Elztalstadion
total_capacity                                            5000.0
lawn_heating                                               False
field_length                                          105.142857
field_width                                            67.493506
link              /sv-waldkirch/spielplan/verein/9487/saison_id/
Name: 35, dtype: object


In [ ]:
import pandas as pd
import numpy as np

# Read both files
teams_df = pd.read_csv("../datasets/final_datasets/final_cleaned_teams.csv")
ratings_df = pd.read_csv("../datasets/teams/stadium_ratings.csv")

print("Teams DataFrame:")
print(teams_df.head(3))
print("\nRatings DataFrame:")
print(ratings_df.head(3))

                       team              stadium_name  total_capacity  \
0         Borussia Dortmund         SIGNAL IDUNA PARK        813650.0   
1             VfL Wolfsburg          Volkswagen Arena        289170.0   
2         FC Bayern München             Allianz Arena        750000.0   
3         Arminia Bielefeld               SchücoArena        273320.0   
4               SC Freiburg       Europa-Park Stadion        347000.0   
5       TSG 1899 Hoffenheim             PreZero Arena        301500.0   
6  Borussia Mönchengladbach  Stadion im Borussia-Park        540420.0   
7       Bayer 04 Leverkusen                  BayArena        302100.0   
8                 VfR Aalen              CENTUS Arena        145000.0   
9       1.FC Kaiserslautern      Fritz-Walter-Stadion        493500.0   

   lawn_heating  field_length  field_width  \
0          True    105.000000    68.000000   
1          True    105.000000    68.000000   
2          True    105.000000    68.000000   
3          T

In [9]:
import pandas as pd
import numpy as np

# Read both files
teams_df = pd.read_csv("../datasets/final_datasets/final_cleaned_teams.csv", encoding="latin1")
ratings_df = pd.read_csv("../datasets/teams/stadium_ratings.csv", encoding="latin1")

In [11]:
# Clean ratings DataFrame
# Remove duplicates based on stadium_name
ratings_df = ratings_df.drop_duplicates(subset=['stadium_name'])

# Convert ratings columns to numeric, replacing any errors with NaN
# Only use columns that exist in ratings_df
expected_columns = ['overall', 'location', 'view', 'facilities', 'food_drink', 'police', 'atmosphere']
rating_columns = [col for col in expected_columns if col in ratings_df.columns]
for col in rating_columns:
    ratings_df[col] = pd.to_numeric(ratings_df[col], errors='coerce')

# Keep only the columns we need for merging
ratings_df = ratings_df[['stadium_name'] + rating_columns]

print("Cleaned Ratings DataFrame:")
print(f"Number of unique stadiums: {len(ratings_df)}")
print("\nSample of cleaned ratings:")
print(ratings_df.head())

Cleaned Ratings DataFrame:
Number of unique stadiums: 157

Sample of cleaned ratings:
          stadium_name  overall  location  view  facilities  police  \
0    SIGNAL IDUNA PARK      4.5       4.2   4.4         4.1     3.6   
1     Volkswagen Arena      3.2       4.5   3.8         4.0     4.3   
2        Allianz Arena      4.2       3.1   4.5         4.2     3.8   
3          SchücoArena      3.1       3.5   3.7         3.3     3.3   
4  Europa-Park Stadion      4.1       NaN   NaN         NaN     NaN   

   atmosphere  
0         4.5  
1         4.3  
2         3.9  
3         3.8  
4         NaN  


In [12]:
# Function to normalize stadium names for better matching
def normalize_stadium_name(name):
    name = str(name).upper()
    # Remove special characters and extra spaces
    name = ''.join(e for e in name if e.isalnum() or e.isspace())
    name = ' '.join(name.split())  # Remove extra spaces
    # Common replacements
    replacements = {
        'STADION': 'STADIUM',
        'ARENA': 'ARENA',
        'PARK': 'PARK',
    }
    for old, new in replacements.items():
        name = name.replace(old, new)
    return name

# Apply normalization to both dataframes
teams_df['stadium_key'] = teams_df['stadium_name'].apply(normalize_stadium_name)
ratings_df['stadium_key'] = ratings_df['stadium_name'].apply(normalize_stadium_name)

# Print a sample of normalized names
print("Sample of normalized stadium names:")
print(pd.DataFrame({
    'Original': teams_df['stadium_name'].head(),
    'Normalized': teams_df['stadium_key'].head()
}))

Sample of normalized stadium names:
              Original          Normalized
0    SIGNAL IDUNA PARK   SIGNAL IDUNA PARK
1     Volkswagen Arena    VOLKSWAGEN ARENA
2        Allianz Arena       ALLIANZ ARENA
3         SchÃ¼coArena        SCHÃ¼COARENA
4  Europa-Park Stadion  EUROPAPARK STADIUM


In [13]:
# Merge dataframes based on normalized stadium names
merged_df = teams_df.merge(ratings_df[['stadium_key'] + rating_columns],
                         on='stadium_key',
                         how='left')

# Handle missing values in ratings columns
# For ratings, we'll use median values as it's more robust than mean for ordinal ratings
for col in rating_columns:
    median_rating = merged_df[col].median()
    merged_df[col] = merged_df[col].fillna(median_rating)

# Drop the temporary key column
merged_df = merged_df.drop('stadium_key', axis=1)

# Reorder columns to have ratings at the end
final_cols = [col for col in merged_df.columns if col not in rating_columns] + rating_columns

# Create final dataframe with ordered columns
final_df = merged_df[final_cols]

print("Final DataFrame Sample:")
print(final_df.head())
print("\nMissing values after handling:")
print(final_df.isnull().sum())

Final DataFrame Sample:
                 team         stadium_name  total_capacity  lawn_heating  \
0   Borussia Dortmund    SIGNAL IDUNA PARK        813650.0          True   
1       VfL Wolfsburg     Volkswagen Arena        289170.0          True   
2  FC Bayern MÃ¼nchen        Allianz Arena        750000.0          True   
3   Arminia Bielefeld         SchÃ¼coArena        273320.0          True   
4         SC Freiburg  Europa-Park Stadion        347000.0          True   

   field_length  field_width  \
0         105.0         68.0   
1         105.0         68.0   
2         105.0         68.0   
3         105.0         68.0   
4         105.0         68.0   

                                                link  overall  location  view  \
0  /borussia-dortmund/spielplan/verein/16/saison_id/      4.5       4.2   4.4   
1      /vfl-wolfsburg/spielplan/verein/82/saison_id/      3.2       4.5   3.8   
2  /fc-bayern-munchen/spielplan/verein/27/saison_id/      4.2       3.1   4.5   
3 

In [16]:
# Save the final dataset
output_path = "../datasets/final_datasets/teams_stadiums_ratings.csv"
final_df.to_csv(output_path, index=False)
print(f"Dataset saved to: {output_path}")

# Display statistics about ratings (0-5 scale)
print("\nRatings Statistics:")
print(final_df[rating_columns].describe())

Dataset saved to: C:\Users\sheha\OneDrive\Desktop\Stadiums\teams_stadiums_ratings.csv

Ratings Statistics:
          overall    location        view  facilities      police  atmosphere
count  159.000000  159.000000  159.000000  159.000000  159.000000  159.000000
mean     2.857233    3.511950    3.933333    3.838994    3.717610    3.740881
std      0.641543    0.508207    0.479935    0.571685    0.594866    0.603284
min      1.000000    1.000000    1.000000    1.500000    0.500000    0.500000
25%      2.700000    3.500000    4.000000    4.000000    3.800000    3.800000
50%      2.900000    3.500000    4.000000    4.000000    3.800000    3.800000
75%      3.000000    3.500000    4.000000    4.000000    3.800000    3.800000
max      4.700000    5.000000    5.000000    5.000000    5.000000    5.000000
